
# Fraud Detection Internship Task

This notebook contains a step-by-step approach to a fraud detection task using transaction data. It covers data loading, exploratory data analysis (EDA), preprocessing, modeling, and evaluation.



## Setup

We set up file paths, constants, and print the data dictionary if available.


In [ ]:

# Setup
DATA_PATH = "Fraud.csv"
DATA_DICT_PATH = "Data Dictionary.txt"
SAMPLE_SIZE = 100000
FULL_RUN = False
RANDOM_SEED = 42

print("DATA_PATH =", DATA_PATH)

import os
if os.path.exists(DATA_DICT_PATH):
    print(open(DATA_DICT_PATH, 'r').read())
else:
    print("Data dictionary not found at", DATA_DICT_PATH)



## Data Loading and Sampling

For memory efficiency, we load the dataset in chunks and sample a manageable subset. If FULL_RUN is True, we load the full dataset.


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

p = Path(DATA_PATH)
peek = pd.read_csv(DATA_PATH, nrows=100)
print("Peek dtypes:\n", peek.dtypes)

if not FULL_RUN:
    chunks = pd.read_csv(DATA_PATH, chunksize=200000)
    parts = []
    for i, chunk in enumerate(chunks):
        if 'isFraud' in chunk.columns:
            frauds = chunk[chunk['isFraud'] == 1]
            nonfraud = chunk[chunk['isFraud'] == 0]
            sample_non = nonfraud.sample(frac=0.02, random_state=RANDOM_SEED) if len(nonfraud) > 0 else nonfraud
            parts.append(pd.concat([frauds, sample_non]))
        else:
            parts.append(chunk.sample(frac=0.02, random_state=RANDOM_SEED))
        if i >= 9: 
            break
    df = pd.concat(parts, ignore_index=True)
    if len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
    print("Quick sample rows:", len(df))
else:
    df = pd.read_csv(DATA_PATH)
    print("Full dataset rows:", len(df))

df.head()



## Exploratory Data Analysis (EDA)

We check the fraud rate, inspect missing values, and visualize the transaction amount distribution.


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if 'isFraud' in df.columns:
    vc = df['isFraud'].value_counts()
    print("Fraud rate: {:.4f}%".format(100 * vc.get(1, 0) / vc.sum()))

print("\nMissing values:")
print(df.isna().mean().sort_values(ascending=False).head(10))

if 'amount' in df.columns:
    plt.figure(figsize=(6, 3))
    sns.histplot(df['amount'].clip(upper=df['amount'].quantile(0.99)), bins=50)
    plt.title("Amount distribution (99% cap)")
    plt.show()



## Preprocessing

- Drop columns with >80% missing values  
- Feature engineering: log transform of amount, hour and day extraction from step, large transaction flag, merchant flags  
- Prepare feature matrix X and target y


In [ ]:

data = df.copy()
drop_cols = data.columns[data.isna().mean() > 0.8].tolist()
if drop_cols: 
    print(f"Dropping columns with >80% missing values: {drop_cols}")
    data = data.drop(columns=drop_cols)

if 'amount' in data.columns:
    data['amount_log'] = np.log1p(data['amount'])
if 'step' in data.columns:
    data['hour'] = data['step'] % 24
    data['day'] = data['step'] // 24
if 'amount' in data.columns:
    data['is_large_txn'] = (data['amount'] > 200000).astype(int)
for c in ['nameOrig', 'nameDest']:
    if c in data.columns:
        data[c + '_is_merchant'] = data[c].astype(str).str.startswith('M').astype(int)

num_cols = data.select_dtypes(include=['number']).columns.tolist()
if 'isFraud' in num_cols:
    num_cols.remove('isFraud')
X = data[num_cols].fillna(0)
y = data['isFraud']



## Train/Validation Split

We create a balanced training set with a max size limit, ensuring stratified split.


In [ ]:

from sklearn.model_selection import train_test_split

MAX_TRAIN = 80000
df_model = X.copy()
df_model['isFraud'] = y.values
fr = df_model[df_model['isFraud'] == 1]
non = df_model[df_model['isFraud'] == 0]
non_samp = non.sample(n=min(len(non), MAX_TRAIN - len(fr)), random_state=RANDOM_SEED)
df_small = pd.concat([fr, non_samp]).sample(frac=1, random_state=RANDOM_SEED)
X_small = df_small.drop(columns=['isFraud'])
y_small = df_small['isFraud']
X_train, X_valid, y_train, y_valid = train_test_split(
    X_small, y_small, test_size=0.2, stratify=y_small, random_state=RANDOM_SEED
)



## Modeling

We train and evaluate two models:
- Logistic Regression with balanced class weights  
- LightGBM with early stopping  
Both models are saved for future use.


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED)
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_valid)[:, 1]
print("LR ROC AUC:", roc_auc_score(y_valid, lr_probs))
print("LR AUPRC:", average_precision_score(y_valid, lr_probs))
joblib.dump(lr, "lr_model.pkl")

import lightgbm as lgb
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
params = {"objective": "binary", "metric": "auc", "is_unbalance": True, "seed": RANDOM_SEED}
bst = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[valid_data],
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)

lgb_probs = bst.predict(X_valid)
print("LGB ROC AUC:", roc_auc_score(y_valid, lgb_probs))
print("LGB AUPRC:", average_precision_score(y_valid, lgb_probs))
bst.save_model("lgb_model.txt")



## Evaluation

We calculate precision-recall curve and precision at top 1% transactions.


In [ ]:

from sklearn.metrics import precision_recall_curve, auc

probs = lgb_probs
precision, recall, thresholds = precision_recall_curve(y_valid, probs)
pr_auc = auc(recall, precision)
print("PR AUC:", pr_auc)

def precision_at_k(y_true, y_score, k=0.01):
    n = max(1, int(len(y_score) * k))
    idx = np.argsort(y_score)[-n:]
    return np.mean(np.array(y_true)[idx])

print("Precision@1%:", precision_at_k(y_valid.values, probs, k=0.01))



## Summary of Results

- **PR AUC (Precision-Recall Area Under Curve):** 0.9907  
  This high score shows our model effectively distinguishes fraud transactions, maintaining a great balance between precision and recall.

- **Precision@1%:** 1.0  
  This means that among the top 1% of transactions flagged by the model as most likely fraud, every single one was actually fraudulent — demonstrating the model's strong precision in prioritizing the riskiest transactions.

These metrics indicate the model is highly effective and can be trusted to detect fraud with minimal false alarms, which is crucial in financial applications.
